# Chronological stacking

This notebook compares a three-model stack against the earlier ensemble of 75% histogram-based gradient boosting (HGB) and 25% CatBoost. The stack adds LightGBM and uses logistic regression to learn how to combine the three predicted probabilities.

The logistic regression is trained on predictions for steps 10–24, with each set of predictions generated by models trained only on preceding time steps. The fitted stack is then evaluated on steps 25–29 and 30–35. These later windows were also examined in earlier experiments, so the results should not be treated as an untouched final holdout.

The final section refits the base models on all labelled transactions (steps 1–35), predicts the unlabelled test set and writes the Kaggle submission file.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT/'data'/'raw'
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
features = [c for c in train if c.startswith('feat_')]
labelled = train.dropna(subset=['label']).copy()
assert labelled.label.isin([0,1]).all()
assert {'time_step','txId'}.issubset(train.columns)
assert all(c in test.columns for c in features)
print('Labelled:',len(labelled),'Test:',len(test),'Features:',len(features))
print('Labelled time:',labelled.time_step.min(),labelled.time_step.max())

Labelled: 31235 Test: 15329 Features: 165
Labelled time: 1 35


In [3]:
SEEDS = [11,42,73,101,202]
HGB_PARAMS = dict(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=42)
CAT_PARAMS = dict(iterations=700,depth=6,learning_rate=0.045,l2_leaf_reg=5,loss_function='Logloss',verbose=False,thread_count=-1)
# LightGBM configuration used in this stacking experiment.
LGB_PARAMS = dict(n_estimators=700,learning_rate=0.045,num_leaves=31,max_depth=-1,
                  reg_lambda=5.0,random_state=42,verbosity=-1,n_jobs=-1)
META_WINDOWS = [(10,14),(15,19),(20,24)]
EVAL_WINDOWS = [(25,29),(30,35)]

def fit_predict_models(tr,va):
    Xtr=tr[features]; ytr=tr.label.astype(int); Xva=va[features]
    if ytr.nunique()!=2: raise ValueError('Training slice has only one class')
    h=HistGradientBoostingClassifier(**HGB_PARAMS).fit(Xtr,ytr)
    ph=h.predict_proba(Xva)[:,1]
    pcs=[]
    for seed in SEEDS:
        c=CatBoostClassifier(**CAT_PARAMS,random_seed=seed)
        c.fit(Xtr,ytr)
        pcs.append(c.predict_proba(Xva)[:,1])
    pc=np.mean(pcs,axis=0)
    l=LGBMClassifier(**LGB_PARAMS).fit(Xtr,ytr)
    pl=l.predict_proba(Xva)[:,1]
    return np.column_stack([ph,pc,pl])

def make_window(lo,hi):
    tr=labelled[labelled.time_step<lo]
    va=labelled[labelled.time_step.between(lo,hi)]
    if tr.empty or va.empty: raise ValueError(f'Empty split: {lo}-{hi}')
    print(f'Window {lo}-{hi}: train {len(tr)}, validation {len(va)}, prevalence {va.label.mean():.3%}',flush=True)
    p=fit_predict_models(tr,va)
    return va.label.to_numpy(dtype=int),p

def score(y,p):
    return average_precision_score(y,p)

def reference(p):
    return .75*p[:,0]+.25*p[:,1]

## Out-of-time predictions for stacking

For each of the three windows (10–14, 15–19 and 20–24), the base models train on preceding labelled transactions. Their predictions form the training data for the logistic regression stacker. This avoids training the stacker on predictions made by base models on their own training rows.

In [4]:
meta_parts=[]; meta_labels=[]; meta_ids=[]
for lo,hi in META_WINDOWS:
    y,p=make_window(lo,hi)
    meta_parts.append(p); meta_labels.append(y); meta_ids.extend([f'{lo}-{hi}']*len(y))
    print('  Exp12:',round(score(y,reference(p)),6),'LightGBM:',round(score(y,p[:,2]),6))
P_meta=np.vstack(meta_parts); y_meta=np.concatenate(meta_labels)
print('OOF meta rows:',len(y_meta),'fraud prevalence:',round(y_meta.mean(),4))

Window 10-14: train 11496, validation 3400, prevalence 14.676%
  Exp12: 0.992753 LightGBM: 0.991833
Window 15-19: train 14896, validation 3093, prevalence 16.360%
  Exp12: 0.98827 LightGBM: 0.988677
Window 20-24: train 17989, validation 5617, prevalence 12.605%
  Exp12: 0.878462 LightGBM: 0.876511
OOF meta rows: 12110 fraud prevalence: 0.1415


## Logistic regression stacker

The stacker uses three inputs: predicted fraud probabilities from HGB, CatBoost and LightGBM. Its coefficients are learned from the out-of-time predictions, with regularisation set to `C=0.1`.

The fitted coefficients are approximately 2.414, 3.413 and 2.713 respectively, with an intercept of −3.826. They are coefficients on the logistic scale, not percentage weights. The score printed on the same data used to fit the stacker is diagnostic only, not an independent validation result.

In [5]:
stacker=LogisticRegression(C=0.1,max_iter=2000,solver='lbfgs')
stacker.fit(P_meta,y_meta)
print('Intercept:',stacker.intercept_)
print('Coefficients [HGB, CatBoost, LightGBM]:',stacker.coef_)
print('OOF in-sample stack PR-AUC (NOT valid evaluation):',round(score(y_meta,stacker.predict_proba(P_meta)[:,1]),6))

Intercept: [-3.82596914]
Coefficients [HGB, CatBoost, LightGBM]: [[2.41373771 3.41327452 2.71307266]]
OOF in-sample stack PR-AUC (NOT valid evaluation): 0.945938


## Chronological validation

The fitted stacker is evaluated on steps 25–29 and 30–35 without refitting it on either evaluation window. For each window, the base models are retrained using only earlier labelled transactions.

The stack improved average precision from 0.993123 to 0.993799 on steps 25–29, and from 0.991554 to 0.992635 on steps 30–35. The improvements are small but positive in both windows. ROC-AUC is included for comparison, while average precision is the primary metric.

In [6]:
rows=[]; heldout={}
for lo,hi in EVAL_WINDOWS:
    y,p=make_window(lo,hi)
    pr=reference(p)
    ps=stacker.predict_proba(p)[:,1]
    heldout[(lo,hi)]=(y,p,pr,ps)
    rows.append(dict(period=f'{lo}-{hi}',n=len(y),prevalence=y.mean(),
                     exp12_pr=score(y,pr),stack_pr=score(y,ps),
                     delta_pr=score(y,ps)-score(y,pr),
                     exp12_roc=roc_auc_score(y,pr),stack_roc=roc_auc_score(y,ps)))
results=pd.DataFrame(rows)
display(results.round(6))
print('Mean PR-AUC baseline:',round(results.exp12_pr.mean(),6))
print('Mean PR-AUC stack:',round(results.stack_pr.mean(),6))
print('Mean delta:',round(results.delta_pr.mean(),6))

Window 25-29: train 23606, validation 2775, prevalence 23.495%
Window 30-35: train 26381, validation 4854, prevalence 15.925%


,period,n,prevalence,exp12_pr,stack_pr,delta_pr,exp12_roc,stack_roc
0,25-29,2775,0.234955,0.993123,0.993799,0.000676,0.997625,0.997780
1,30-35,4854,0.159250,0.991554,0.992635,0.001081,0.998122,0.998414


Mean PR-AUC baseline: 0.992338
Mean PR-AUC stack: 0.993217
Mean delta: 0.000879


## Model comparison

The stack improves average precision on both later validation windows, with a mean gain of 0.000879 over the earlier HGB–CatBoost blend. This supported submitting it as a challenger, although the reused validation periods limit how strongly we can interpret the gains. The resulting submission achieved a public Kaggle score of 0.94914, compared with 0.94756 for the earlier blend.

In [7]:
IMPROVES_BOTH = bool((results.delta_pr>0).all())
print('Stack improves both later windows:',IMPROVES_BOTH)
if not IMPROVES_BOTH:
    print('The stack did not improve both validation windows.')
else:
    print('The stack improved both validation windows.')

Stack improves both later windows: True
The stack improved both validation windows.


## Kaggle test predictions

The base models are refitted on all available labelled transactions before predicting `test.csv`. The logistic regression stacker retains the coefficients learned from the earlier out-of-time predictions. The submission uses the column order in `sample_submission.csv` and is saved to `submissions/experiment18_stacking.csv`.

The test labels are unavailable locally; the public leaderboard score is an external evaluation of the submitted predictions.

In [9]:
MAKE_SUBMISSION=True
if MAKE_SUBMISSION:
    if not IMPROVES_BOTH:
        raise RuntimeError('Stack did not improve both later windows; submission blocked.')
    # Test rows do not have labels; fit_predict_models needs only feature columns on va.
    P_test=fit_predict_models(labelled,test)
    predictions=stacker.predict_proba(P_test)[:,1]
    assert len(predictions)==len(test)==len(sample)
    assert list(sample.columns)==['index','target'], f'Unexpected schema: {list(sample.columns)}'
    submission=sample.copy()
    submission['target']=predictions
    assert submission.target.notna().all() and submission.target.between(0,1).all()
    output=ROOT/'submissions'/'experiment18_stacking.csv'
    output.parent.mkdir(parents=True,exist_ok=True)
    submission.to_csv(output,index=False)
    print('Saved:',output)
else:
    print('Submission generation is disabled.')

Saved: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/submissions/experiment18_stacking.csv


## Notes on evaluation

The stacking procedure uses chronological out-of-time predictions to reduce training leakage. It is not nested cross-validation, and the validation periods were reused during the broader experimentation process. The public leaderboard result should therefore be interpreted separately from local validation performance.